# lltPretrainData, Turn Raw Web Pages Into Pretraining Data

LLM Training Lab, module 9. Steps 1 to 7 run on a CPU on the first 60 MB of one Common Crawl WARC file. Our recorded runs are in `courses/llm-training/runs/lltPretrainData/`.

In [ ]:
# Runtime check. Run this first. It prints the fix for each problem it finds.
import platform, shutil, sys
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("FIX: use Python 3.11 or later. On Colab, Runtime > Change runtime type, keep the default runtime.")
try:
    import psutil
    print(f"RAM {psutil.virtual_memory().total / 1e9:.1f} GB")
except ImportError:
    pass
free = shutil.disk_usage(".").free / 1e9
print(f"Free disk {free:.1f} GB")
if free < 2:
    print("FIX: steps 1 to 7 download about 0.4 GB. Free some disk space.")
print("Steps 1 to 7 run on a CPU. No GPU is needed.")

Install the three libraries this lab adds. NumPy, pandas, scikit-learn and PyTorch come with Colab and Kaggle, and this notebook never installs NumPy.

In [ ]:
%pip install -q "trafilatura==2.3.1" "warcio==1.8.2" "fasttext-numpy2==0.10.4"

The filters, PII scrub, decontamination and MinHash code, as one file the steps import.

In [ ]:
%%writefile ptd.py
"""lltPretrainData shared library: the filters, PII scrub, decontamination and MinHash dedup the lab
page builds step by step. Plain Python and NumPy, so the same functions run in the notebook, in the
Kaggle data-prep kernel and (the NumPy parts) in the browser.

Rule thresholds follow Rae et al. 2021 (Gopher, Appendix A, Table A1) and Raffel et al. 2020 (C4,
section 2.2), with the defaults datatrove 0.10.1 uses for the same rules. Words are split on
whitespace here; datatrove splits with a language tokenizer, so a few borderline documents differ
(the measured agreement is in the Run Library file lltPretrainData-cpu-funnel).
"""
import hashlib
import re
from collections import Counter

import numpy as np

# ----------------------------------------------------------------------------------------------
# Gopher quality rules (Rae et al. 2021, Appendix A)
STOP_WORDS = {"the", "be", "to", "of", "and", "that", "have", "with"}
PUNCT = set("!\"#$%&'()*+,-./:;<=>?@[]^_`{|}~")


def gopher_quality(text):
    """Return None if the document passes, else the name of the first rule it fails."""
    words = text.split()
    if not words:
        return "empty"
    content = [w for w in words if any(c not in PUNCT for c in w)]
    if len(content) < 50:
        return "too_few_words"
    if len(content) > 100_000:
        return "too_many_words"
    mean_len = sum(len(w) for w in content) / len(content)
    if mean_len < 3 or mean_len > 10:
        return "mean_word_length"
    if text.count("#") / len(words) > 0.1:
        return "hash_ratio"
    if (text.count("...") + text.count("…")) / len(words) > 0.1:
        return "ellipsis_ratio"
    lines = text.splitlines() or [text]
    if sum(l.lstrip().startswith(("•", "-")) for l in lines) / len(lines) > 0.9:
        return "bullet_lines"
    if sum(l.rstrip().endswith(("...", "…")) for l in lines) / len(lines) > 0.3:
        return "ellipsis_lines"
    if sum(any(c.isalpha() for c in w) for w in words) / len(words) < 0.8:
        return "alpha_words"
    if len(STOP_WORDS & set(w.lower() for w in words)) < 2:
        return "stop_words"
    return None


# ----------------------------------------------------------------------------------------------
# Gopher repetition rules (Rae et al. 2021, Table A1)
TOP_NGRAM = ((2, 0.20), (3, 0.18), (4, 0.16))
DUP_NGRAM = ((5, 0.15), (6, 0.14), (7, 0.13), (8, 0.12), (9, 0.11), (10, 0.10))


def _dups(items):
    seen, n_dup, chars = set(), 0, 0
    for x in items:
        if x in seen:
            n_dup += 1
            chars += len(x)
        else:
            seen.add(x)
    return n_dup, chars


def _dup_ngram_chars(words, n):
    # Characters covered by n-grams already seen, skipping ahead past each repeat (as datatrove does).
    seen, chars, i = set(), 0, 0
    while i < len(words) - n + 1:
        g = " ".join(words[i:i + n])
        if g in seen:
            chars += sum(len(w) for w in words[i:i + n])
            i += n
        else:
            seen.add(g)
            i += 1
    return chars


def gopher_repetition(text):
    if not text.strip():
        return "empty"
    paras = re.split("\n{2,}", text.strip())
    n_dup, chars = _dups(paras)
    if n_dup / len(paras) > 0.30:
        return "dup_paragraphs"
    if chars / len(text) > 0.20:
        return "dup_paragraph_chars"
    lines = re.split("\n+", text)
    n_dup, chars = _dups(lines)
    if n_dup / len(lines) > 0.30:
        return "dup_lines"
    if chars / len(text) > 0.20:
        return "dup_line_chars"
    words = text.split()
    for n, frac in TOP_NGRAM:
        grams = [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
        if grams:
            g, c = Counter(grams).most_common(1)[0]
            if len(g) * c / len(text) > frac:
                return "top_%d_gram" % n
    for n, frac in DUP_NGRAM:
        if _dup_ngram_chars(words, n) / len(text) > frac:
            return "dup_%d_gram" % n
    return None


# ----------------------------------------------------------------------------------------------
# C4 rules (Raffel et al. 2020, section 2.2), as the paper states them: keep lines that end in
# terminal punctuation and have at least 5 words, drop pages with fewer than 3 sentences.
# datatrove 0.10.1's C4QualityFilter defaults swap those two numbers (5 sentences, 3 words).
# The bad-words list is not applied here.
POLICY = ("terms of use", "privacy policy", "cookie policy", "uses cookies", "use of cookies", "use cookies")
CITATION = re.compile(r"\[\d*\]|\[edit\]|\[citation needed\]")
SENT_END = re.compile(r"[.!?][\"')\]]*(?=\s|$)")


def c4_clean(text, terminal_punct=True):
    """Return (cleaned_text, None) if the page is kept, else (None, rule)."""
    kept, sentences = [], 0
    for line in text.splitlines():
        line = line.strip()
        words = line.split()
        if any(len(w) > 1000 for w in words):
            continue
        line = CITATION.sub("", line)
        if terminal_punct and (not line.endswith((".", "?", "!", '"', "'")) or line.endswith("...")):
            continue
        if len(words) < 5:
            continue
        low = line.lower()
        if "lorem ipsum" in low:
            return None, "lorem_ipsum"
        if "javascript" in low:
            continue
        if "{" in line:
            return None, "curly_bracket"
        if any(p in low for p in POLICY):
            continue
        sentences += max(1, len(SENT_END.findall(line)))
        kept.append(line)
    if sentences < 3:
        return None, "fewer_than_3_sentences"
    return "\n".join(kept), None


# ----------------------------------------------------------------------------------------------
# PII scrub: email addresses and public IPv4 addresses, replaced by fixed placeholders.
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
IPV4 = re.compile(r"\b(?:(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\.){3}(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\b")


def _private_ip(ip):
    a, b = (int(x) for x in ip.split(".")[:2])
    return a in (0, 10, 127) or (a == 192 and b == 168) or (a == 172 and 16 <= b <= 31) or (a == 169 and b == 254) or a >= 224


def scrub_pii(text):
    """Return (text, n_emails, n_ips) with emails and public IPv4s replaced."""
    text, n_email = EMAIL.subn("<EMAIL>", text)
    n_ip = 0

    def ip_sub(m):
        nonlocal n_ip
        if _private_ip(m.group(0)):
            return m.group(0)
        n_ip += 1
        return "<IP>"

    text = IPV4.sub(ip_sub, text)
    return text, n_email, n_ip


# ----------------------------------------------------------------------------------------------
# n-gram decontamination: drop a document that shares any n-gram (13 words, lower-cased) with an
# evaluation set.
WORD = re.compile(r"[a-z0-9]+")


def ngrams(text, n=13):
    w = WORD.findall(text.lower())
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}


def build_eval_index(eval_texts, n=13):
    idx = set()
    for t in eval_texts:
        idx |= ngrams(t, n)
    return idx


def contaminated(text, index, n=13):
    return not ngrams(text, n).isdisjoint(index)


# ----------------------------------------------------------------------------------------------
# Exact dedup on normalised text
def norm_text(text):
    return " ".join(WORD.findall(text.lower()))


def exact_key(text):
    return hashlib.sha1(norm_text(text).encode()).hexdigest()


# ----------------------------------------------------------------------------------------------
# MinHash + LSH in NumPy. Shingles are word 5-grams of the normalised text, hashed to 31 bits.
# Each of the K hash functions is h(x) = (a * x + b) mod P with P = 2**31 - 1, so a * x stays
# below 2**62 and fits in uint64.
P31 = np.uint64(2**31 - 1)


def shingles(text, n=5):
    w = norm_text(text).split()
    grams = {" ".join(w[i:i + n]) for i in range(max(1, len(w) - n + 1))} if w else set()
    return np.array([int.from_bytes(hashlib.blake2b(g.encode(), digest_size=4).digest(), "little") & 0x7FFFFFFF
                     for g in grams], dtype=np.uint64)


def make_hashes(k, seed=1):
    rng = np.random.default_rng(seed)
    a = rng.integers(1, 2**31 - 1, size=k, dtype=np.uint64)
    b = rng.integers(0, 2**31 - 1, size=k, dtype=np.uint64)
    return a, b


def minhash(sh, a, b):
    if sh.size == 0:
        return np.full(a.size, 2**31 - 1, dtype=np.uint64)
    return ((sh[:, None] * a[None, :] + b[None, :]) % P31).min(axis=0)


def lsh_candidates(sigs, bands, rows):
    """sigs: (n_docs, bands*rows). Return the set of (i, j) pairs that share at least one band."""
    pairs = set()
    for bnd in range(bands):
        buckets = {}
        for i, s in enumerate(sigs[:, bnd * rows:(bnd + 1) * rows]):
            buckets.setdefault(s.tobytes(), []).append(i)
        for ids in buckets.values():
            for x in range(len(ids)):
                for y in range(x + 1, len(ids)):
                    pairs.add((ids[x], ids[y]))
    return pairs


def p_candidate(j, bands, rows):
    """Probability that two documents with Jaccard similarity j share at least one band."""
    return 1 - (1 - j ** rows) ** bands


def clusters(n, pairs):
    """Union-find over candidate pairs. Returns the root of each document; keep one per root."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    return [find(i) for i in range(n)]


def jaccard(s1, s2):
    s1, s2 = set(s1.tolist()), set(s2.tolist())
    return len(s1 & s2) / max(1, len(s1 | s2))


## Step 1. Download one Common Crawl file and extract the text

In [ ]:
import os
import time
import urllib.request
import trafilatura
from warcio.archiveiterator import ArchiveIterator

URL = ("https://data.commoncrawl.org/crawl-data/CC-MAIN-2024-10/segments/1707947473347.0/"
       "warc/CC-MAIN-20240220211055-20240221001055-00000.warc.gz")
for attempt in range(7):  # Common Crawl answers 503 when busy, so wait and retry
    if os.path.exists("warc_slice.gz") and os.path.getsize("warc_slice.gz") == 62914560:
        break
    try:
        req = urllib.request.Request(URL, headers={"Range": "bytes=0-62914559", "User-Agent": "llt-lab"})
        with urllib.request.urlopen(req, timeout=300) as r:
            data = r.read()
        with open("warc_slice.gz", "wb") as f:
            f.write(data)
    except Exception as e:
        print("download failed,", e, "| retrying in", 15 * 2 ** attempt, "s")
        time.sleep(15 * 2 ** attempt)

records, pages = 0, []
try:
    with open("warc_slice.gz", "rb") as f:
        for rec in ArchiveIterator(f):
            records += 1
            h = rec.http_headers
            is_html = h is not None and "html" in (h.get_header("Content-Type") or "").lower()
            if rec.rec_type == "response" and is_html and h.get_statuscode() == "200":
                url = rec.rec_headers.get_header("WARC-Target-URI")
                pages.append((url, rec.content_stream().read().decode("utf-8", "replace")))
except Exception as e:  # the first 60 MB ends in the middle of a record
    print("slice ends mid-record,", type(e).__name__)

docs = []
for url, html in pages:
    text = trafilatura.extract(html, favor_precision=True, include_comments=False, deduplicate=True)
    if text and text.strip():
        docs.append((url, text))
print("WARC records", records, "| HTML pages with status 200", len(pages), "| pages with text", len(docs))
print("HTML characters", sum(len(h) for _, h in pages), "| text characters", sum(len(t) for _, t in docs))


## Step 2. Keep English pages with fastText language ID

In [ ]:
import collections
import urllib.request
import fasttext

urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "lid.176.bin")
lid = fasttext.load_model("lid.176.bin")

def language(text):
    labels, probs = lid.predict(text.replace(chr(10), " ")[:20000], k=1)
    return labels[0].replace("__label__", ""), float(probs[0])

langs = [language(text) for _, text in docs]
print(collections.Counter(code for code, _ in langs).most_common(6))
english = [d for d, (code, p) in zip(docs, langs) if code == "en" and p >= 0.65]
unsure = sum(code == "en" and p < 0.65 for code, p in langs)
print("English with score >= 0.65", len(english), "| English below 0.65", unsure)


## Step 3. Write the Gopher quality rules

The same cell runs in the lab page.

In [ ]:
STOP = {"the", "be", "to", "of", "and", "that", "have", "with"}

def gopher_quality(text):
    words = text.split()
    content = [w for w in words if any(c.isalnum() for c in w)]
    if not 50 <= len(content) <= 100_000:
        return "word count outside 50 to 100,000"
    mean = sum(len(w) for w in content) / len(content)
    if not 3 <= mean <= 10:
        return "mean word length outside 3 to 10"
    if (text.count("#") + text.count("...")) / len(words) > 0.1:
        return "too many # or ..."
    if sum(any(c.isalpha() for c in w) for w in words) / len(words) < 0.8:
        return "under 80% of words have a letter"
    if len(STOP & {w.lower() for w in words}) < 2:
        return "fewer than 2 stop words"
    return None

def top_ngram_share(text, n):
    words = text.split()
    grams = [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
    best = max(set(grams), key=lambda g: (grams.count(g), len(g)))
    return len(best) * grams.count(best) / len(text)

story = ("The library opens at nine every morning and closes at six. Children come after school "
         "to read, and older readers sit near the back with the newspapers. On rainy days the "
         "reading room is full, and the staff put out extra chairs for everyone. The building "
         "was restored last year with a new roof and better lighting over the long tables.")
pages = {
    "article": story,
    "menu": "Home About Shop Contact Login Cart Search Sale New Brands",
    "price list": " ".join(f"item{i} 4.99 USD 12 pcs" for i in range(30)),
    "keyword spam": " ".join(["buy the cheap phone cases and save today"] * 9),
    "tag cloud": " ".join(["#deal #sale #shop the best of"] * 12),
}
for name, text in pages.items():
    rule = gopher_quality(text)
    share = top_ngram_share(text, 2)
    if rule is None and share > 0.20:
        rule = "top 2-gram covers %.0f%% of characters" % (100 * share)
    print(f"{name:13} {'kept' if rule is None else 'removed, ' + rule}")


## Step 4. Run the Gopher and C4 rules over every page

In [ ]:
import collections
import ptd  # the notebook writes ptd.py in its first cells

removed = collections.Counter()
kept = []
for url, text in english:
    rule = ptd.gopher_quality(text) or ptd.gopher_repetition(text)
    cleaned = None
    if rule is None:
        cleaned, rule = ptd.c4_clean(text)
    if rule:
        removed[rule] += 1
    else:
        kept.append((url, cleaned))
for rule, n in removed.most_common():
    print(f"{rule:24} {n:5}")
print("kept", len(kept), "of", len(english))
words_in = sum(len(t.split()) for _, t in english)
words_out = sum(len(t.split()) for _, t in kept)
print("words kept", words_out, "of", words_in)


## Step 5. Train a quality classifier on pages scored by Llama 3

In [ ]:
import urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score, roc_auc_score

URL = ("https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu-llama3-annotations/"
       "resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet")
urllib.request.urlretrieve(URL, "edu_scores.parquet")
df = pd.read_parquet("edu_scores.parquet", columns=["text", "score"])
perm = np.random.default_rng(0).permutation(len(df))
test, train = perm[: len(df) // 6], perm[len(df) // 6:]

hv = HashingVectorizer(n_features=2**20, alternate_sign=False, ngram_range=(1, 2), norm=None)
tf = TfidfTransformer(sublinear_tf=True)
X_train = tf.fit_transform(hv.transform(df["text"].iloc[train]))
X_test = tf.transform(hv.transform(df["text"].iloc[test]))
y_train, y_test = df["score"].values[train], df["score"].values[test]
model = Ridge(alpha=1.0).fit(X_train, y_train)
pred = model.predict(X_test)

print("rows", len(df), "| share of test pages scored 3 or more", round(float(np.mean(y_test >= 3)), 4))
print("mean squared error", round(float(np.mean((pred - y_test) ** 2)), 4),
      "| predicting the mean", round(float(np.mean((y_train.mean() - y_test) ** 2)), 4))
print("AUC for score >= 3", round(roc_auc_score(y_test >= 3, pred), 4))
for cut in (3.0, 2.0):
    print(f"keep pred >= {cut}: F1 {f1_score(y_test >= 3, pred >= cut):.4f}, pages kept {int((pred >= cut).sum())}")


## Step 6. Remove near-duplicates with MinHash and LSH

In [ ]:
import hashlib
import numpy as np

P = 2**31 - 1
rng = np.random.default_rng(1)
K, BANDS, ROWS = 112, 14, 8
A = rng.integers(1, P, size=K, dtype=np.uint64)
B = rng.integers(0, P, size=K, dtype=np.uint64)

def shingles(text, n=5):
    w = "".join(c.lower() if c.isalnum() else " " for c in text).split()
    grams = {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}
    return np.array([int(hashlib.sha1(g.encode()).hexdigest()[:8], 16) % P for g in grams], dtype=np.uint64)

def minhash(sh):
    return ((sh[:, None] * A[None, :] + B[None, :]) % P).min(axis=0)

def jaccard(s1, s2):
    s1, s2 = set(s1.tolist()), set(s2.tolist())
    return len(s1 & s2) / len(s1 | s2)

base = ("The library opens at nine every morning and closes at six in the evening. Children come "
        "after school to read stories, and older readers sit near the back with the newspapers. "
        "On rainy days the reading room is full, and the staff put out extra chairs. The building "
        "was restored last year with a new roof and better lighting over the long tables. ")
others = {
    "same page, new footer": base + "Copyright 2024. All rights reserved.",
    "same page, one sentence changed": base.replace("extra chairs", "folding chairs and tables"),
    "different page": "Preheat the oven to 180 degrees. Mix two cups of flour with one egg, a cup of milk "
                      "and a pinch of salt. Pour the batter into a greased tin and bake it for forty minutes.",
}
s0 = shingles(base + "Contact the front desk for opening hours.")
m0 = minhash(s0)
for name, text in others.items():
    s1 = shingles(text)
    m1 = minhash(s1)
    agree = m0 == m1
    bands_hit = sum(agree[b * ROWS:(b + 1) * ROWS].all() for b in range(BANDS))
    print(f"{name:32} Jaccard {jaccard(s0, s1):.3f}  MinHash estimate {agree.mean():.3f}  bands matched {bands_hit}")

print("P(candidate) = 1 - (1 - J**8)**14")
for J in [0.3, 0.5, 0.7, 0.8, 0.9]:
    print(f"  J = {J:.1f}   {1 - (1 - J**ROWS)**BANDS:.4f}")


In [ ]:
import numpy as np
import ptd

texts = [t for _, t in kept]
seen, unique = set(), []
for i, t in enumerate(texts):
    key = ptd.exact_key(t)
    if key not in seen:
        seen.add(key)
        unique.append(i)
print("exact duplicates removed", len(texts) - len(unique))

a, b = ptd.make_hashes(112, seed=1)
sigs = np.stack([ptd.minhash(ptd.shingles(texts[i]), a, b) for i in unique])
pairs = ptd.lsh_candidates(sigs, bands=14, rows=8)
roots = ptd.clusters(len(unique), pairs)
keep = [unique[k] for k in range(len(unique)) if roots[k] == k]
deduped = [kept[i] for i in keep]
print("candidate pairs", len(pairs), "| near duplicates removed", len(unique) - len(keep), "| pages left", len(deduped))
for x, y in sorted(pairs)[:3]:
    j = ptd.jaccard(ptd.shingles(texts[unique[x]]), ptd.shingles(texts[unique[y]]))
    print(f"pair {x}-{y}  exact Jaccard {j:.3f}  signature agreement {float((sigs[x] == sigs[y]).mean()):.3f}")


## Step 7. Decontaminate against benchmarks and scrub personal data

Predict first. How many of 20 reworded ARC questions does a 13-gram check catch?

In [ ]:
import urllib.request
import pandas as pd
import ptd

HF = "https://huggingface.co/datasets/"
urllib.request.urlretrieve(HF + "allenai/ai2_arc/resolve/refs%2Fconvert%2Fparquet/ARC-Easy/test/0000.parquet", "arc.parquet")
urllib.request.urlretrieve(HF + "Rowan/hellaswag/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet", "hs.parquet")
arc, hs = pd.read_parquet("arc.parquet"), pd.read_parquet("hs.parquet")
eval_texts = [q + " " + " ".join(c["text"]) for q, c in zip(arc["question"], arc["choices"])]
eval_texts += [c + " " + e[int(l)] for c, e, l in zip(hs["ctx"], hs["endings"], hs["label"])]
index = ptd.build_eval_index(eval_texts, n=13)
print("evaluation items", len(eval_texts), "| distinct 13-grams", len(index))

clean = [(u, t) for u, t in deduped if not ptd.contaminated(t, index)]
print("pages sharing a 13-gram with ARC-Easy or HellaSwag", len(deduped) - len(clean))

emails = ips = 0
scrubbed = []
for url, text in clean:
    text, n_email, n_ip = ptd.scrub_pii(text)
    emails += n_email
    ips += n_ip
    scrubbed.append((url, text))
print("emails replaced", emails, "| public IPv4 addresses replaced", ips)

question = arc.loc[arc["id"] == "Mercury_182158", "question"].iloc[0] + " wash hands"
paraphrase = "A pupil has just finished a lab exercise. Before walking out of the laboratory, what should the pupil do last? Wash their hands."
print("verbatim caught", ptd.contaminated("Practice question. " + question, index),
      "| paraphrase caught", ptd.contaminated("Practice question. " + paraphrase, index))


## Band checks

In [ ]:
# Band checks against our recorded run (lltPretrainData-t1-notebook, one CPU process).
# Counts can move by a page or two with another trafilatura version.
print("pages with text", len(docs), "| English", len(english), "| after rules", len(kept), "| after dedup", len(deduped))
assert abs(len(docs) - 3118) <= 3, "step 1 count differs. Check that the slice is the first 62,914,560 bytes."
assert abs(len(english) - 846) <= 3, "step 2 count differs. Check the 0.65 threshold."
assert abs(len(kept) - 372) <= 3, "step 4 count differs. Check ptd.py."
print("All band checks passed")

In [ ]:
# Peak RAM of this notebook so far
import psutil, sys
mi = psutil.Process().memory_info()
peak_b = getattr(mi, "peak_wset", None)
if peak_b is None:
    import resource
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_b = r * 1024 if sys.platform.startswith("linux") else r
print(f"Peak RAM {peak_b / 1e9:.2f} GB")

## Steps 8 to 10. Training runs on a T4

These steps train 18 small GPTs. Our runs used two Kaggle sessions: a CPU session for the data prep (`prep_ptd.py`, 35 min on 4 cores) and a 2x T4 session for the training queues (`worker_ptd.py`). Both scripts are in this folder of the repository, with the queues in `kaggle/cells_ptd_train.py`. Every result is in `runs/lltPretrainData/`.

The cells below run on a CPU: the extraction test from step 9, the token accounting and the RegMix regression from step 10.

In [ ]:
import torch

@torch.no_grad()
def extraction(model, canaries, eot, prefix=50, suffix=50):
    """canaries: LongTensor [n, prefix + suffix]. Greedy-decode suffix tokens after EOT + the
    first prefix tokens. A canary is extracted when every generated token matches."""
    model.eval()
    start = torch.full((canaries.size(0), 1), eot, dtype=torch.long, device=canaries.device)
    x = torch.cat([start, canaries[:, :prefix]], dim=1)
    for _ in range(suffix):
        next_id = model(x)[:, -1, :].argmax(dim=-1, keepdim=True)
        x = torch.cat([x, next_id], dim=1)
    match = x[:, 1 + prefix:] == canaries[:, prefix:prefix + suffix]
    return match.all(dim=1).float().mean().item(), match.float().mean().item()

# A check you can run on a CPU. A "model" that has stored the canaries extracts all of them.
torch.manual_seed(0)
V, EOT = 4096, 0
canaries = torch.randint(1, V, (4, 100))
table = {}
for row in canaries.tolist():
    for i in range(1, 100):
        table[tuple(row[:i])] = row[i]

def lookup_model(x):
    out = torch.zeros(x.size(0), x.size(1), V)
    for r, seq in enumerate(x.tolist()):
        out[r, -1, table.get(tuple(seq[1:]), 1)] = 1.0
    return out

lookup_model.eval = lambda: None
print("extraction rate, token match:", extraction(lookup_model, canaries, EOT))


In [ ]:
import numpy as np

# Measured on our corpora (run lltPretrainData-cpu-corpora): mean tokens per document,
# and the unique tokens we tokenized, for web (filtered C4), Wikipedia (wikitext-103) and stories.
names = ["web", "wikipedia", "stories"]
mean_doc_tokens = np.array([922.6, 5578.2, 248.2])
unique_tokens = np.array([55_141_258, 60_004_232, 39_712_508])
budget = 300 * 64 * 512  # the mixture runs, 300 steps of 64 x 512 tokens

def token_share_from_doc_weights(doc_w, mean_len):
    t = doc_w * mean_len
    return t / t.sum()

def effective_epochs(token_w, budget, unique):
    return token_w * budget / unique

equal = np.array([1 / 3, 1 / 3, 1 / 3])
by_docs = token_share_from_doc_weights(equal, mean_doc_tokens)
for n, s in zip(names, by_docs):
    print(f"equal weights by document -> {n:9} gets {s:.1%} of the tokens")
for n, e in zip(names, effective_epochs(equal, budget, unique_tokens)):
    print(f"equal weights by token    -> {n:9} is seen {e:.3f} times")


In [ ]:
import numpy as np

# Our mixture runs: token weights for web, Wikipedia, stories, and the measured bits per byte
# on each held-out set (runs lltPretrainData-T2-mix-*, 300 steps of 64 x 512 tokens, 11M parameters).
RUNS = [
    ([0.3333, 0.3333, 0.3333], [2.0557, 1.9416, 1.1410]),  # equal by tokens
    ([0.1345, 0.8288, 0.0367], [2.0825, 1.8012, 1.4800]),  # equal by documents
    ([0.6, 0.2, 0.2], [2.0169, 1.9794, 1.2241]),           # web 60%
    ([0.2, 0.6, 0.2], [2.0550, 1.8344, 1.2113]),           # Wikipedia 60%
    ([0.2, 0.2, 0.6], [2.1246, 2.0278, 1.0603]),           # stories 60%
]
W = np.array([r[0] for r in RUNS])          # [n_runs, 3] mixture weights by tokens
Y = np.array([r[1] for r in RUNS])          # [n_runs, 3] bits per byte on web, wiki, stories

def fit_ridge(W, y, lam=1e-3):
    X = np.hstack([W, np.ones((len(W), 1))])
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

def predict(coef, w):
    return np.append(w, 1.0) @ coef

names = ["web", "wiki", "stories"]
for d in range(3):
    errs = []
    for k in range(len(W)):             # leave one run out, fit on the rest, predict it
        rest = [i for i in range(len(W)) if i != k]
        errs.append(predict(fit_ridge(W[rest], Y[rest, d]), W[k]) - Y[k, d])
    print(f"{names[d]:8} leave-one-out error, mean absolute {np.mean(np.abs(errs)):.3f} bits per byte")
coef = [fit_ridge(W, Y[:, d]) for d in range(3)]
w_new = np.array([0.5, 0.3, 0.2])
print("prediction for web 0.5, wiki 0.3, stories 0.2:", [round(float(predict(c, w_new)), 3) for c in coef])
